In [16]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [17]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/UGC-409_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/UGC-409_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     361   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        42   (2094, 964)   float32   


In [18]:
image_cut = image[55:175, 0:1890]
image_error_cut = image_error[55:175, 0:1890]

In [19]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.01736
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [20]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) #+
        #sersic_1d(x_hr, params6) #+
        #sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [21]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/PSF/Real_seeing_UGC409.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 120, 1)

In [22]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choc_area = single_integral(x, params5, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area+ choc_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(blue_area)
    print(green_area)
    print(purple_area)
    print(orange_area)
    print(choc_area)
    print('Sum',sum_area)
    print('Total', total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area, (choc_area*100)/sum_area ]        

## OIII

In [23]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit_OIII = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/Fit/OIII_fit.csv", index_col = 0)

In [24]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -sigma, fit_OIII['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -sigma, fit_OIII['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -sigma, fit_OIII['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -sigma, fit_OIII['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -sigma, fit_OIII['Component 5'].iloc[3] + sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -sigma, fit_OIII['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_1s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

1.519916778433179
0.22831589136016722
8.039885600346849e-06
0.008090867047437932
5.245028029962271e-55
Sum 1.7563315767263847
Total 1.680263297756812
0.06176984036141288
0.43536866338794333
0.0009758202866904705
0.04891349713405638
1.0950425093104825e-21
Sum 0.5470278211701031
Total 0.5470688838097458
0.034140249858046395
0.41393437851134185
4.947532646295108
0.22398033571670029
5.0041340626509414e-11
Sum 5.619587610431239
Total 4.509442802815986
0.020323490378697122
0.23593898578728245
0.013595297038194133
0.44518931923933724
1.9747075649473155e-06
Sum 0.7150490671510759
Total 0.7131310118726288
0.004436775672368701
0.004558409647504238
2.050151344324126e-06
0.01935035172652111
0.21977784112614634
Sum 0.24812542832388473
Total 0.24818870381928382
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       8.653928e+01  1.129190e+01  6.075223e-01   2.842251   1.788118
Green      1.299959e+01  7.958803e+01  7.365921e+00  32.996195   1.837139
Magenta    4.577658e

In [25]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -2*sigma, fit_OIII['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -2*sigma, fit_OIII['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -2*sigma, fit_OIII['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -2*sigma, fit_OIII['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -2*sigma, fit_OIII['Component 5'].iloc[3] + 2*sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -2*sigma, fit_OIII['Component 6'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

2.6013063042467257
0.45948484687498703
1.711943801113056e-05
0.01648654210613882
9.500068483706294e-51
Sum 3.0772948126658624
Total 2.9641822988755218
0.161388131165174
1.14657058028665
0.006039020426729619
0.1400229868495198
9.633524192401477e-19
Sum 1.4540207187280731
Total 1.4536992762845262
0.06895696137782072
0.8268066663030013
7.28199042234083
0.4569174034618276
7.776748125522553e-10
Sum 8.634671454261156
Total 7.000170866972642
0.040944048521408316
0.4746487706764261
0.11635612693650278
0.8416441907235668
1.2054702668381393e-05
Sum 1.4736051915605721
Total 1.4554034860025928
0.008904563222198047
0.009513318287954684
4.287625963087788e-06
0.03943627898764524
0.4356097210066517
Sum 0.49346816913041275
Total 0.4936131784625313
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       8.453224e+01  1.109944e+01  7.986055e-01   2.778495   1.804486
Green      1.493145e+01  7.885517e+01  9.575427e+00  32.210037   1.927848
Magenta    5.563145e-04  4.153325e-01

In [26]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -3*sigma, fit_OIII['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -3*sigma, fit_OIII['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -3*sigma, fit_OIII['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -3*sigma, fit_OIII['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -3*sigma, fit_OIII['Component 5'].iloc[3] + 3*sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'].iloc[3] -3*sigma, fit_OIII['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

3.2645940505297544
0.6961828114627366
2.8493224597151596e-05
0.025503553934815557
9.565913163132123e-47
Sum 3.9863089091519033
Total 3.8636159943108517
0.24942027649500037
1.6910363589262394
0.03908227454511179
0.21680069772536026
6.177772768383805e-17
Sum 2.196339607691712
Total 2.188112731283692
0.10516045509073561
1.2373476229284939
7.815611480559224
0.7068481112042884
9.571729014893598e-09
Sum 9.864967679354471
Total 8.113080600609901
0.06216889058273352
0.7187185221842525
0.7311550274196126
1.1728973733028702
6.04331264548412e-05
Sum 2.6850002466159233
Total 2.5541685174164455
0.013434805465208102
0.015281040825020598
6.926011544876732e-06
0.06102210267013504
0.6418152569336464
Sum 0.731560131905555
Total 0.73180826897061
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       8.189516e+01  1.135618e+01  1.065999e+00   2.315415   1.836459
Green      1.746435e+01  7.699339e+01  1.254285e+01  26.767913   2.088829
Magenta    7.147771e-04  1.779428e+00  7.

## HBeta

In [27]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC409/UGC409_1/FORS2.2021-10-05T04_55_53.494/Fit/Hbeta_fit.csv", index_col = 0)

In [28]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 1'].iloc[3] -sigma, fit_OIII['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 2'].iloc[3] -sigma, fit_OIII['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 3'].iloc[3] -sigma, fit_OIII['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 4'].iloc[3] -sigma, fit_OIII['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 5'].iloc[3] -sigma, fit_OIII['Component 5'].iloc[3] + sigma)

#data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_OIII['Component 6'].iloc[3] -sigma, fit_OIII['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

1.175140920264233
0.18597088051844718
0.0
0.0068603332647866515
1.481651541958323e-66
Sum 1.3679721340474669
Total 1.4353507473117442
0.0370662759066574
0.4276460212402511
1.573397492653239e-24
0.026147251057259156
2.20983800896771e-24
Sum 0.49085954820416766
Total 0.4908763415747852
0.018499312508584283
0.41239547974838
24.081276613973294
0.13202560967637594
6.419871308264205e-12
Sum 24.644197015913054
Total 4.282798577156424
0.010348118903373421
0.2343389439768216
0.027151080930288507
0.4509740024103728
8.167828769481814e-07
Sum 0.7228129630037333
Total 0.6778767193317312
0.001870694396059978
0.006280475998085328
0.0
0.012828210782163967
0.1691731702414002
Sum 0.19015255141770945
Total 0.1901963898868548
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       8.590386e+01  7.551300e+00  7.506559e-02   1.431645   0.983786
Green      1.359464e+01  8.712187e+01  1.673398e+00  32.420412   3.302862
Magenta    0.000000e+00  3.205392e-22  9.771581e+01   3.756308

In [29]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 1'].iloc[3] -2*sigma, fit_OIII['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 2'].iloc[3] -2*sigma, fit_OIII['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 3'].iloc[3] -2*sigma, fit_OIII['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 4'].iloc[3] -2*sigma, fit_OIII['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_OIII['Component 5'].iloc[3] -2*sigma, fit_OIII['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

2.0242457342460995
0.3751400566267672
0.0
0.013859414177025558
5.129388978797247e-61
Sum 2.4132452050498925
Total 2.509822535658123
0.09679050221091276
1.1275696534164088
0.00503841743654845
0.07458971072294403
7.343286636936159e-21
Sum 1.3039882837868142
Total 1.2997395203784268
0.037445305983996174
0.8242656370098107
35.1605643859756
0.278947975443382
1.3958170115621654e-10
Sum 36.301223304552366
Total 6.553396943570443
0.02088171275361979
0.471638521244202
0.4305812869660743
0.8003475009615622
5.823928418989808e-06
Sum 1.7234548458538774
Total 1.3254035607065302
0.003757639756112581
0.012983374353108198
0.0
0.025979126763871093
0.33574141992285267
Sum 0.37846156079594456
Total 0.37856467935085963
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       8.388065e+01  7.422651e+00  1.031516e-01   1.211619   0.992872
Green      1.554505e+01  8.647084e+01  2.270628e+00  27.365876   3.430566
Magenta    0.000000e+00  3.863852e-01  9.685779e+01  24.983613   0.00

In [30]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 1'].iloc[3] -3*sigma, fit_OIII['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 2'].iloc[3] -3*sigma, fit_OIII['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 3'].iloc[3] -3*sigma, fit_OIII['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 4'].iloc[3] -3*sigma, fit_OIII['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 5'].iloc[3] -3*sigma, fit_OIII['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

2.543795831814288
0.5705812467695262
7.385153e-317
0.021140939662386636
7.725834718116649e-56
Sum 3.135518018246201
Total 3.2366099794774117
0.15047803356908354
1.6559603011137196
0.11929892826866423
0.11569737843805297
1.0032615812470703e-18
Sum 2.0414346413895204
Total 1.9391624385863553
0.057309612890690864
1.2347154037486185
37.50645277535806
0.4610009179506278
2.2881102034843345e-09
Sum 39.25947871223611
Total 7.513892600483645
0.031793407273160434
0.7146880948385541
3.2392796189813593
1.0432718889266757
3.3350188585998045e-05
Sum 5.029066360208335
Total 2.2503421005719737
0.005677348292941738
0.020547961163061346
0.0
0.03979102804525358
0.49569421347410747
Sum 0.5617105509753642
Total 0.5618923957732295
                  Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue        8.112841e+01  7.371190e+00  1.459765e-01   0.632193   1.010725
Green       1.819735e+01  8.111748e+01  3.145012e+00  14.211149   3.658105
Magenta    2.355321e-315  5.843877e+00  9.553477e+01  64